# RoadVision_DUET: High Resolution Object Detection for Dense Traffic 🚗🛵

This notebook details the training and inference pipeline for the DUET Hackathon competition. The objective is to accurately detect and classify 13 diverse vehicle types in complex and dense traffic environments of Dhaka Streets. 

## 1. Environment Setup
First, we install the necessary Ultralytics framework for YOLOv8 and verify our GPU acceleration to ensure optimal training speeds.

In [ ]:
!pip install -q ultralytics ensemble-boxes

import torch
import ultralytics

print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device Name: {torch.cuda.get_device_name(0)}")
ultralytics.checks()

In [ ]:
import os
import pandas as pd
from tqdm import tqdm

CSV_PATH = "/kaggle/input/competitions/road-vision/RoadVision_DUET/train/train.csv"
BASE_DIR = "dataset"
DIRS = [
    f"{BASE_DIR}/images/train",
    f"{BASE_DIR}/images/val",
    f"{BASE_DIR}/labels/train",
    f"{BASE_DIR}/labels/val"
]

for folder in DIRS:
    os.makedirs(folder, exist_ok=True)

df = pd.read_csv(CSV_PATH)
grouped = df.groupby('image_id')

for image_id, group in tqdm(grouped):
    base_filename = os.path.splitext(str(image_id))[0]
    label_path = f"{BASE_DIR}/labels/train/{base_filename}.txt"
    with open(label_path, "w") as f:
        for _, row in group.iterrows():
            class_id = int(row['class_id'])
            x_c = row['x_center']
            y_c = row['y_center']
            w = row['width']
            h = row['height']
            f.write(f"{class_id} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f}\n")

## 2. Data Engineering & Formatting
The provided hackathon dataset contains annotations in a single `.csv` file. To train a YOLO model, we need to restructure this data.

**Pipeline steps:**
1. Group bounding box coordinates by `image_id`.
2. Convert the coordinates into YOLO's normalized `.txt` format.
3. Establish a strict 80/20 train-validation split to monitor for overfitting.
4. Automatically generate the `dataset.yaml` configuration file required by Ultralytics.

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

IMAGE_SOURCE_DIR = "/kaggle/input/competitions/road-vision/RoadVision_DUET/train/images"
IMAGE_EXT = ".jpg"

CLASS_NAMES = [
    "Rickshaw",
    "Motorcycle",
    "Tempu",
    "Sedan Car",
    "Pickup",
    "Microbus",
    "Mini Bus",
    "Mini Truck",
    "Agro Use",
    "Medium Truck",
    "Large Bus",
    "Heavy Truck",
    "Trailer"
]

all_labels = [f for f in os.listdir("dataset/labels/train") if f.endswith('.txt')]
random.seed(42)
random.shuffle(all_labels)

val_count = int(len(all_labels) * 0.2)
val_labels = all_labels[:val_count]

for label_file in val_labels:
    shutil.move(
        os.path.join("dataset/labels/train", label_file),
        os.path.join("dataset/labels/val", label_file)
    )

missing_images = 0
for split in ['train', 'val']:
    labels_in_split = os.listdir(f"dataset/labels/{split}")
    for label_file in tqdm(labels_in_split, desc=f"Copying {split} images"):
        image_filename = label_file.replace(".txt", IMAGE_EXT)
        source_img_path = os.path.join(IMAGE_SOURCE_DIR, image_filename)
        dest_img_path = os.path.join(f"dataset/images/{split}", image_filename)
        if os.path.exists(source_img_path):
            shutil.copy(source_img_path, dest_img_path)
        else:
            missing_images += 1

yaml_names = "\n".join([f"  {i}: {name}" for i, name in enumerate(CLASS_NAMES)])
yaml_content = f"""
path: /kaggle/working/dataset
train: images/train
val: images/val

names:
{yaml_names}
"""
with open("dataset.yaml", "w") as f:
    f.write(yaml_content.strip())

## 3. Model Architecture & Optimization
For this environment I selected the **YOLOv8x (Extra-Large)** architecture. Dense street scenes feature highly overlapping objects, unpredictable occlusion and small vehicles in the background. The massive parameter count of the `x` variant is necessary to extract these finegrained, complex spatial features.

**Training & Augmentation Strategy:**
* **High-Resolution Input:** Scaled the image size to `1280px` to retain critical details of smaller objects in the distance.
* **Aggressive Augmentation:** To make the model robust against the chaotic nature of the roads, I applied heavy spatial and color distortions. Setting `mosaic=1.0` and `mixup=0.15` forces the network to learn partial vehicle shapes, preventing it from over-relying on perfect visibility.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8x.pt")

results = model.train(
    data="/kaggle/working/dataset.yaml",
    epochs=200,
    patience=30,
    imgsz=1280,
    batch=4,
    project="/kaggle/working/training",
    name="run_x_1280",
    lr0=0.01,
    lrf=0.001,
    warmup_epochs=5,
    cos_lr=True,
    weight_decay=0.0005,
    optimizer="AdamW",
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    degrees=5.0,
    translate=0.1,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    copy_paste=0.1,
    workers=2
)

## 4. Final Inference & Submission Generation
With the model optimized, we deploy it against the unseen test set. 

To squeeze out maximum accuracy for the competition metric, I utilized **Test-Time Augmentation (TTA)** (`augment=True`). By evaluating flipped and scaled versions of the test matrices simultaneously, the model's confidence across edge cases improves. 

I applied a custom confidence threshold of `0.10` and an Intersection over Union (IoU) threshold of `0.4` for Non-Maximum Suppression (NMS). This mathematical balance ensures we aggressively capture valid objects while merging overlapping bounding box predictions into single, clean detections.

In [ ]:
from ultralytics import YOLO
import os
import pandas as pd
from tqdm import tqdm

TEST_DIR = "/kaggle/input/competitions/road-vision/RoadVision_DUET/test/images"
MODEL_PATH = "/kaggle/working/training/run_x_1280/weights/best.pt"

model = YOLO(MODEL_PATH)
test_imgs = sorted(os.listdir(TEST_DIR))

rows = []
for img_name in tqdm(test_imgs):
    img_path = os.path.join(TEST_DIR, img_name)
    bare_id = img_name.replace('.jpg', '').replace('.png', '')
    prediction_string = ""
    results = model.predict(
        source=img_path,
        conf=0.10,
        iou=0.4,
        augment=True,
        verbose=False,
        device=0
    )
    for r in results:
        for box in r.boxes:
            c_id = int(box.cls[0].item())
            conf = float(box.conf[0].item())
            x_c, y_c, w, h = box.xywhn[0].tolist()
            prediction_string += f"{c_id} {conf:.4f} {x_c:.6f} {y_c:.6f} {w:.6f} {h:.6f} "
    final_str = prediction_string.strip() or " "
    rows.append({'image_id': bare_id, 'PredictionString': final_str})

final = pd.DataFrame(rows)
final.to_csv('/kaggle/working/submission_v2.csv', index=False, encoding='utf-8')
print("Done!", len(final))